In [0]:
"""Construct a complete, linked synthetic Pharos CDM dataset.

Loaded after the support and MOSTLY AI helpers by the main notebook. The
fallback imports keep this file directly importable outside Databricks.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date

import pandas as pd
from pharos_cdm.enums.cancer_history import CancerType
from pharos_cdm.enums.followup import (
    DistantMetastasis,
    LocalRecurrence,
    VitalStatus,
)
from pharos_cdm.enums.medical_history import FamilyHistory, GeneticTesting, GermlineMutation
from pharos_cdm.enums.pathology_tumour_focus import (
    HER2Score,
    InSituType,
    InvasivePresent,
    MicroInvasionPresence,
    ReceptorStatus,
)
from pharos_cdm.enums.tnm import PathologicalTStage
from pharos_cdm.enums.treatment import TherapyOngoing, TreatmentIntent, TreatmentType
from pharos_cdm.refs.treatment import TREATMENT_REFERENCE
from pharos_cdm.validation.cross_table.medical_history import FAMILY_HISTORY_FLAGS

try:
    from synth_pharos_support import (
        GROUP_CODE,
        IdentityMinter,
        TableSpec,
        Timeline,
        build_registry,
        det_rng,
        is_tnm_field,
        make_timeline,
        may_null,
        mint_pharosids,
        null_inapplicable,
        sample_ethnicity_pair,
        sample_field,
        sample_tnm,
        topo_order,
    )
    from synth_pharos_mostly import MostlyDonorPool, coerce_donor_value
except ImportError:
    pass  # Names already exist when this file is loaded with Databricks %run.

TODAY = date(2026, 8, 18)
ONE_PER_PERSON = {"followup", "medical_history"}

CHILD_COUNTS: dict[str, tuple[float, int]] = {
    "cohort": (1.2, 3),
    "person_tumour_group": (0.15, 2),
    "family_cancer_history": (0.8, 4),
    "personal_cancer_history": (0.3, 2),
    "comorbidity": (1.5, 6),
    "medication": (2.5, 8),
    "medical_history": (1.0, 1),
    "imaging": (2.0, 6),
    "tumour": (1.0, 2),
    "pathology": (1.2, 3),
    "sample": (1.5, 5),
    "treatment": (2.0, 6),
    "followup": (1.0, 1),
    "local_recurrence": (0.25, 2),
    "distant_metastasis": (0.2, 2),
    "pathology_focus": (1.3, 4),
}


@dataclass
class Dataset:
    registry: dict[str, TableSpec]
    tables: dict[str, pd.DataFrame]
    seed: int
    persons: dict[str, int]
    prof: dict | None
    mostly_data: dict[str, pd.DataFrame] | None = None


def _n_children(
    rng,
    table: str,
    group: str,
    prof: dict | None,
    donor_count: int | None = None,
) -> int:
    expected, cap = CHILD_COUNTS.get(table, (1.0, 3))
    if donor_count is not None:
        donor_cap = max(cap, 40 if table == "pathology" else cap)
        return min(max(int(donor_count), 0), donor_cap)
    try:
        histogram = prof["tables"][table]["seqlen"][group]["counts_hist"]
        values = [int(key) for key in histogram]
        weights = [histogram[str(value)] for value in values]
        total = sum(weights)
        return int(rng.choice(values, p=[weight / total for weight in weights]))
    except (KeyError, TypeError):
        return min(int(rng.poisson(expected)), cap)


def _profile_codes(prof: dict | None, table: str, field: str, group: str):
    try:
        return prof["tables"][table]["columns"][field]["by_group"][group]["codes"]
    except (KeyError, TypeError):
        return None


def _mint_non_identity_pk(spec: TableSpec, minter: IdentityMinter):
    field = next(item for item in spec.fields if item.name == spec.pk)
    value = minter.next(spec.name)
    if (field.meta.sql_type or "").upper() in ("BIGINT", "INT", "SMALLINT"):
        return value
    return f"SYN-{spec.name.upper().replace('_', '-')}-{value}"


def _postprocess_row(
    row: dict,
    spec: TableSpec,
    timeline: Timeline,
    group: str,
    rng,
) -> dict:
    table = spec.name
    if "date_checked" in row:
        row["date_checked"] = timeline.end.isoformat()

    if table == "person":
        row["dob"] = timeline.dob.isoformat()

    elif table == "tumour":
        row["date_of_diagnosis"] = timeline.dx.isoformat()
        if "age_at_diagnosis" in row:
            row["age_at_diagnosis"] = int((timeline.dx - timeline.dob).days / 365.25)
        # Conditional fields are always populated safely; TNM applicability is
        # already enforced by the sampler.
        for field in spec.fields:
            if is_tnm_field(field) and row.get(field.name) is None:
                row[field.name] = sample_tnm(rng, field.enum_cls, group)
        from pharos_cdm.enums.tumour import NeoAdjuvantIndication

        if NeoAdjuvantIndication(row["neoadjuvant_indication"]).name.startswith("YES"):
            if row.get("c_ajcc_edition") is None:
                fs = next(item for item in spec.fields if item.name == "c_ajcc_edition")
                row["c_ajcc_edition"] = sample_field(rng, table, fs, timeline, group)
        inflammatory = next((f for f in spec.fields if f.name == "disease_inflammatory"), None)
        if inflammatory and row.get("disease_inflammatory") is not None:
            from pharos_cdm.enums.tumour import DiseaseInflammatory

            if DiseaseInflammatory(row["disease_inflammatory"]) is DiseaseInflammatory.YES:
                valid_t4 = [
                    member
                    for member in next(f for f in spec.fields if f.name == "clinical_t_stage").enum_cls
                    if member.name.startswith("T4")
                    and group
                    in __import__(
                        "pharos_cdm.refs.tnm", fromlist=["cancer_type_map"]
                    ).cancer_type_map(type(member)).get(member.name, set())
                ]
                if valid_t4:
                    row["clinical_t_stage"] = valid_t4[int(rng.integers(0, len(valid_t4)))].value

    elif table == "imaging":
        event = timeline.post_dx(rng)
        row["image_date"] = event.isoformat()
        row["days_diagnosis_imaging"] = (event - timeline.dx).days

    elif table == "pathology":
        event = timeline.post_dx(rng)
        row["surgery_date"] = event.isoformat()
        row["days_diagnosis_surgery"] = (event - timeline.dx).days
        row["age_at_surgery"] = int((event - timeline.dob).days / 365.25)
        row["total_nodes_removed"] = max(int(row.get("total_nodes_removed") or 5), 1)
        row["total_positive_nodes"] = min(
            max(int(row.get("total_positive_nodes") or 0), 0),
            row["total_nodes_removed"],
        )
        for field in spec.fields:
            if is_tnm_field(field) and row.get(field.name) is None:
                row[field.name] = sample_tnm(rng, field.enum_cls, group)

    elif table == "pathology_focus":
        invasive = InvasivePresent(row["invasive_present"])
        if group != "breast":
            # Breast-only conditional fields cannot satisfy the flat-model
            # invasive rule for other groups, so keep those foci non-invasive.
            row["invasive_present"] = InvasivePresent.ABSENT.value
            invasive = InvasivePresent.ABSENT
            row["microinvasion"] = MicroInvasionPresence.ABSENT.value
            row["er_status"] = ReceptorStatus.UNKNOWN.value
            row["pr_status"] = ReceptorStatus.UNKNOWN.value
            row["her2_status"] = None
        if invasive is InvasivePresent.PRESENT:
            for field in ("morphology", "invasive_size_path", "grade", "total_size_path"):
                if row.get(field) is None:
                    fs = next(item for item in spec.fields if item.name == field)
                    row[field] = sample_field(rng, table, fs, timeline, group)
        for status_field, score_field in (
            ("er_status", "er_score_sample"),
            ("pr_status", "pr_score_sample"),
            ("her2_status", "her2_score_sample"),
        ):
            status = row.get(status_field)
            if status is not None and ReceptorStatus(status) is not ReceptorStatus.UNKNOWN:
                if row.get(score_field) is None:
                    fs = next(item for item in spec.fields if item.name == score_field)
                    row[score_field] = sample_field(rng, table, fs, timeline, group)
        if row.get("her2_score_sample") == HER2Score.TWO_PLUS.value and row.get("her2_fish") is None:
            fs = next(item for item in spec.fields if item.name == "her2_fish")
            row["her2_fish"] = sample_field(rng, table, fs, timeline, group)
        insitu_type = row.get("insitu_type")
        if row.get("dcis_subtype") is not None and (
            insitu_type is None
            or InSituType(insitu_type) not in (InSituType.DCIS, InSituType.DCIS_LCIS)
        ):
            row["dcis_subtype"] = None
        if row.get("lcis_subtype") is not None and (
            insitu_type is None
            or InSituType(insitu_type) not in (InSituType.LCIS, InSituType.DCIS_LCIS)
        ):
            row["lcis_subtype"] = None

    elif table == "sample":
        from pharos_cdm.enums.sample import TissueType

        requires = {
            TissueType.TUMOUR,
            TissueType.NORMAL_TISSUE,
            TissueType.TUMOUR_BED,
            TissueType.METASTATIC_TUMOUR,
            TissueType.BENIGN_TISSUE,
            TissueType.NON_TUMOUR_TISSUE,
            TissueType.METASTATIC_LYMPH_NODE,
            TissueType.NON_TUMOUR_LYMPH_NODE,
        }
        if TissueType(row["tissue_type"]) in requires and row.get("tumour_sample") is None:
            fs = next(item for item in spec.fields if item.name == "tumour_sample")
            row["tumour_sample"] = sample_field(rng, table, fs, timeline, group)
        if "sample_date" in row:
            row["sample_date"] = timeline.post_dx(rng).isoformat()

    elif table == "treatment":
        treatment_type = TreatmentType(row["treatment_type"])
        no_drug = {TreatmentType.NONE, TreatmentType.UNKNOWN}
        refs = [ref for ref in TREATMENT_REFERENCE if ref.treatment_type is treatment_type]
        row["treatment_name"] = None
        row["treatment_regimen"] = None
        if treatment_type not in no_drug | {TreatmentType.RADIOTHERAPY, TreatmentType.OTHER}:
            singles = [ref for ref in refs if not ref.is_regimen]
            regimens = [ref for ref in refs if ref.is_regimen]
            if regimens and rng.random() < 0.35:
                row["treatment_regimen"] = regimens[int(rng.integers(0, len(regimens)))].name
                if singles:
                    row["treatment_name"] = singles[int(rng.integers(0, len(singles)))].name
            elif singles:
                row["treatment_name"] = singles[int(rng.integers(0, len(singles)))].name
            elif regimens:
                row["treatment_regimen"] = regimens[int(rng.integers(0, len(regimens)))].name
        if treatment_type in no_drug:
            row["treatment_intent"] = TreatmentIntent.NOT_APPLICABLE.value
        elif row.get("treatment_intent") is None:
            row["treatment_intent"] = TreatmentIntent.UNKNOWN.value
        start = timeline.post_dx(rng)
        start_days = (start - timeline.dx).days
        row["treatment_start_date"] = start.isoformat()
        row["days_diagnosis_treatment_start"] = start_days
        if row.get("therapy_ongoing") is None:
            row["therapy_ongoing"] = TherapyOngoing.UNKNOWN.value
        if (
            treatment_type
            not in {
                TreatmentType.RADIOTHERAPY,
                TreatmentType.ENDOCRINE,
                TreatmentType.OTHER,
                TreatmentType.NONE,
                TreatmentType.UNKNOWN,
            }
            and row.get("treatment_cycles") is None
        ):
            row["treatment_cycles"] = int(rng.integers(1, 13))
        ongoing = TherapyOngoing(row["therapy_ongoing"]) is TherapyOngoing.YES
        if ongoing:
            for field in (
                "treatment_end_date",
                "days_diagnosis_treatment_end",
                "treatment_duration",
                "treatment_end_reason",
            ):
                row[field] = None
        elif treatment_type not in no_drug:
            end = timeline.between(rng, start, timeline.end)
            end_days = (end - timeline.dx).days
            row["treatment_end_date"] = end.isoformat()
            row["days_diagnosis_treatment_end"] = end_days
            row["days_diagnosis_treatment_last_given"] = end_days
            row["treatment_duration"] = end_days - start_days
        else:
            for field in (
                "treatment_end_date",
                "days_diagnosis_treatment_end",
                "days_diagnosis_treatment_last_given",
                "treatment_duration",
                "treatment_cycles",
            ):
                row[field] = None

    elif table == "followup":
        row["vital_status"] = (
            VitalStatus.DECEASED.value if timeline.death else VitalStatus.ALIVE.value
        )
        row["date_of_death"] = timeline.death.isoformat() if timeline.death else None
        row["date_of_last_followup"] = timeline.end.isoformat()
        years = (timeline.end - timeline.dx).days / 365.25
        row["years_lastfollowup"] = round(years, 3)
        row["years_diagnosistodeath"] = round(years, 3) if timeline.death else None
        if not timeline.death:
            row["cause_of_death"] = None

    elif table == "medical_history":
        if GeneticTesting(row["genetic_testing"]) is GeneticTesting.YES and row.get(
            "any_germline_mutation"
        ) is None:
            row["any_germline_mutation"] = GermlineMutation.UNKNOWN.value

    elif table == "local_recurrence":
        event = timeline.post_dx(rng)
        row["recurrence_date"] = event.isoformat()
        row["days_diagnosis_recurrence"] = (event - timeline.dx).days

    elif table == "distant_metastasis":
        event = timeline.post_dx(rng)
        if "metastasis_date" in row:
            row["metastasis_date"] = event.isoformat()
        if "days_diagnosis_metastasis" in row:
            row["days_diagnosis_metastasis"] = (event - timeline.dx).days

    return null_inapplicable(row, spec, group)


def _row(
    rng,
    spec: TableSpec,
    timeline: Timeline,
    group: str,
    minter: IdentityMinter,
    fk_values: dict[str, object],
    prof: dict | None,
    donor_row: dict | None = None,
) -> dict:
    row: dict[str, object] = {}
    for field in spec.fields:
        if field.name in fk_values:
            row[field.name] = fk_values[field.name]
        elif field.meta.is_pk:
            row[field.name] = (
                minter.next(spec.name)
                if field.meta.is_identity
                else _mint_non_identity_pk(spec, minter)
            )
        elif is_tnm_field(field):
            donated = coerce_donor_value(field, donor_row.get(field.name)) \
                if donor_row and field.name in donor_row else None
            pool = [donated] if donated is not None else _profile_codes(
                prof, spec.name, field.name, group)
            has_donor = donor_row is not None and field.name in donor_row
            row[field.name] = (
                sample_tnm(rng, field.enum_cls, group, profiled_codes=pool)
                if donated is not None or not may_null(field, group) or (
                    not has_donor and rng.random() < 0.8)
                else None
            )
        elif donor_row is not None and field.name in donor_row:
            donated = coerce_donor_value(field, donor_row[field.name])
            row[field.name] = donated if donated is not None or may_null(field, group) else sample_field(
                rng, spec.name, field, timeline, group, prof)
        elif may_null(field, group) and prof is None and rng.random() < 0.25:
            row[field.name] = None
        else:
            row[field.name] = sample_field(
                rng, spec.name, field, timeline, group, prof
            )
    now = TODAY.isoformat() + "T00:00:00"
    for audit_column in ("created_at", "updated_at"):
        if audit_column in row:
            row[audit_column] = now
    return _postprocess_row(row, spec, timeline, group, rng)


def _parent_target_col(spec: TableSpec) -> str:
    for column, table, target_column in spec.entity_fks:
        if spec.parent and (column, table) == spec.parent:
            return target_column
    return "pharosid"


def _derive_flags(rows: dict[str, list[dict]], registry: dict[str, TableSpec]) -> None:
    recurrence_ids = {row["pharosid"] for row in rows["local_recurrence"]}
    metastasis_ids = {row["pharosid"] for row in rows["distant_metastasis"]}
    for row in rows["followup"]:
        row["local_recurrence"] = (
            LocalRecurrence.YES.value
            if row["pharosid"] in recurrence_ids
            else LocalRecurrence.NO.value
        )
        row["distant_metastasis"] = (
            DistantMetastasis.YES.value
            if row["pharosid"] in metastasis_ids
            else DistantMetastasis.NO.value
        )

    reported: dict[str, set[CancerType]] = {}
    for row in rows["family_cancer_history"]:
        try:
            cancer = CancerType(row.get("cancer_type"))
        except (ValueError, TypeError):
            continue
        reported.setdefault(row["pharosid"], set()).add(cancer)
    for row in rows["medical_history"]:
        cancers = reported.get(row["pharosid"], set())
        for flag in FAMILY_HISTORY_FLAGS:
            has_match = bool(cancers) if flag.cancer_types is None else any(
                cancer in cancers for cancer in flag.cancer_types
            )
            row[flag.flag_field] = (
                FamilyHistory.YES.value if has_match else FamilyHistory.NO.value
            )

    foci_by_pathology: dict[object, list[dict]] = {}
    for focus in rows["pathology_focus"]:
        foci_by_pathology.setdefault(focus["pathology_id"], []).append(focus)
    for pathology in rows["pathology"]:
        foci = foci_by_pathology.get(pathology["pathology_id"], [])
        has_micro = any(
            MicroInvasionPresence(focus["microinvasion"])
            is MicroInvasionPresence.PRESENT
            for focus in foci
        )
        if has_micro:
            for focus in foci:
                focus["invasive_present"] = InvasivePresent.ABSENT.value
            pathology["pathological_t_stage"] = PathologicalTStage.T1mi.value


def generate_dataset(
    seed: int,
    persons: dict[str, int],
    prof: dict | None = None,
    mostly_data: dict[str, pd.DataFrame] | None = None,
) -> Dataset:
    registry = build_registry()
    order = topo_order(registry)
    minter = IdentityMinter()
    rows: dict[str, list[dict]] = {table: [] for table in registry}
    by_person: dict[str, dict[str, list[dict]]] = {table: {} for table in registry}
    timelines: dict[str, Timeline] = {}
    groups: dict[str, str] = {}
    donors = MostlyDonorPool(mostly_data, seed) if mostly_data else None

    def remember(table: str, row: dict) -> None:
        rows[table].append(row)
        by_person[table].setdefault(row["pharosid"], []).append(row)

    for group, count in persons.items():
        if group not in GROUP_CODE:
            raise ValueError(f"unknown tumour group: {group}")
        for pid in mint_pharosids(group, count):
            rng = det_rng(seed, f"person:{pid}")
            timeline = make_timeline(rng, TODAY)
            timelines[pid], groups[pid] = timeline, group
            row = _row(
                rng,
                registry["person"],
                timeline,
                group,
                minter,
                {"pharosid": pid, "tumour_group": GROUP_CODE[group]},
                prof,
                donors.next_row("person", pid, group) if donors else None,
            )
            try:
                from pharos_cdm.refs.ethnicity import parent_ethnicity_code
                valid_ethnicity = (
                    row.get("ethnic_group") is None
                    or parent_ethnicity_code(row["ethnic_group"]) == row.get("ethnicity")
                )
            except (ValueError, TypeError):
                valid_ethnicity = False
            if not valid_ethnicity or row.get("ethnicity") is None:
                ethnicity, ethnic_group = sample_ethnicity_pair(rng)
                row["ethnicity"], row["ethnic_group"] = ethnicity, ethnic_group
            remember("person", row)

    for table in order[1:]:
        spec = registry[table]
        if spec.parent is None:
            raise AssertionError(f"unexpected root: {table}")
        fk_column, parent = spec.parent
        target_column = _parent_target_col(spec)
        secondary = [
            (column, target, target_col)
            for column, target, target_col in spec.entity_fks
            if (column, target) != spec.parent
        ]
        secondary_mandatory = {
            column: next(field for field in spec.fields if field.name == column).meta.mandatory
            for column, _, _ in secondary
        }
        for parent_row in rows[parent]:
            pid = parent_row["pharosid"]
            group, timeline = groups[pid], timelines[pid]
            parent_pk = parent_row[registry[parent].pk]
            rng = det_rng(seed, f"{table}:{parent_pk}")
            donor_count = None
            if donors and parent in {"person", "followup"}:
                donor_count = donors.count(table, pid, group)
            count = 1 if table in ONE_PER_PERSON else _n_children(
                rng, table, group, prof, donor_count=donor_count)
            for _ in range(count):
                fk_values = {fk_column: parent_row[target_column], "pharosid": pid}
                wired = True
                for column, target, target_col in secondary:
                    pool = by_person[target].get(pid, [])
                    if pool:
                        fk_values[column] = pool[int(rng.integers(0, len(pool)))][target_col]
                    elif secondary_mandatory[column]:
                        wired = False
                        break
                    else:
                        fk_values[column] = None
                if wired:
                    remember(
                        table,
                        _row(
                            rng,
                            spec,
                            timeline,
                            group,
                            minter,
                            fk_values,
                            prof,
                            donors.next_row(table, pid, group) if donors else None,
                        ),
                    )

    _derive_flags(rows, registry)
    tables = {
        table: pd.DataFrame(
            table_rows,
            columns=[field.name for field in registry[table].fields],
        )
        for table, table_rows in rows.items()
    }
    return Dataset(registry, tables, seed, dict(persons), prof, mostly_data)